[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Foundations of Signal Processing (2)

The sequel [Part 1](./Foundations_of_Signal_Processing_1.ipynb) promised: the z-transform as the discrete world's native language, multirate processing (changing sample rates without lying), the polyphase trick that makes it cheap, and a first meeting with wavelets.

## 1. Pre-requisites

- [Part 1](./Foundations_of_Signal_Processing_1.ipynb) Sessions 3–8.
- [Complex Analysis Lite](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb) — poles, ROC, residues (used throughout Session 1).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The z-Transform & ROC* (~35 min)
**Goal:** master the discrete transform: ROC geometry, stability, and inversion by partial fractions.
**Builds on:** [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S4–S5; [Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb). &nbsp; **Feeds into:** Session 2 (multirate).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The z-Transform & ROC</b></summary>

**Timing (~35 min).** 8 min z as "the DTFT with a volume knob" · 12 min the ROC and what it encodes · 10 min the demo · 5 min the stability/causality trade.

**Board first — the one-line relationship.** On $z = re^{j\omega}$, the z-transform is the DTFT of $x[n]r^{-n}$. So $r$ is a *damping knob*: signals too wild for Fourier to converge become tame once multiplied by a decaying exponential, and $r = 1$ recovers the DTFT exactly. That makes the z-transform a generalisation the room already half-owns rather than a new object.

**The ROC is the session's real content — insist it is part of the answer.** Students write $X(z) = z/(z-a)$ and stop. That expression alone does *not* determine a signal: with $|z| > |a|$ it is a causal growing sequence, with $|z| < |a|$ an anticausal decaying one. Two signals, one formula. The ROC is not a footnote appended to the transform; it is half of it.

**Then the two rules that make it useful.** Stability ⟺ the ROC contains the unit circle. Causality ⟺ the ROC extends outward from the outermost pole. Have the room combine them for a pole *outside* the circle: the two conditions cannot both hold, so you must choose. That is exactly the demo below, and it is the cleanest motivation for why ROC bookkeeping matters.

**Ask the room.** "So can a system with a pole at 1.25 be used at all?" Yes — but only non-causally, meaning offline processing where the future is available. That is precisely `filtfilt` from [Filter Design](./Filter_Design.ipynb), whose backward pass is a deliberately anticausal filter, legal only because the whole signal is already recorded. An apparently abstract ROC distinction turns out to be the line between real-time and offline.

**Point at the delay property.** $x[n-k] \leftrightarrow z^{-k}X(z)$ is why every digital filter is a polynomial in $z^{-1}$: $z^{-1}$ *is* "delay by one sample." Once that clicks, a transfer function stops being algebra and becomes a parts list — coefficients are multipliers, powers of $z^{-1}$ are delay elements, and the block diagram reads straight off the formula.

**If the room has done [Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb)**, note that inversion by partial fractions is residue calculus and the table is derivable. If not, take the table on trust — the ROC reasoning is what this session needs.
</details>

## 2. The z-Transform

💡 **Intuition.** The z-transform $X(z) = \sum_n x[n] z^{-n}$ is the DTFT with a volume knob: on $z = re^{j\omega}$, it's the DTFT of $x[n] r^{-n}$ — signals too wild for Fourier become tame after exponential damping. The **ROC** records which damping levels work, and it carries real information: the *same* algebraic $X(z)$ with different ROCs describes different signals (causal vs anticausal). Stability = ROC contains the unit circle; causality = ROC extends outward.

**The table you can now derive** (via residues, [Complex Analysis S2](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb)):

| $x[n]$ | $X(z)$ | ROC |
|---|---|---|
| $\delta[n]$ | $1$ | all $z$ |
| $a^n u[n]$ | $\frac{z}{z-a}$ | $|z| > |a|$ |
| $-a^n u[-n-1]$ | $\frac{z}{z-a}$ | $|z| < |a|$ ← same formula, different signal! |
| $r^n \sin(\theta n) u[n]$ | ratio with poles $re^{\pm j\theta}$ | $|z| > r$ |

**Key properties:** delay $x[n-k] \leftrightarrow z^{-k}X(z)$ (why filters are polynomials in $z^{-1}$), convolution ↔ multiplication.

In [2]:
# The ROC is not decoration: one X(z), two signals — only the ROC disambiguates
a = 1.25                                       # pole OUTSIDE the unit circle
n_ax = np.arange(-20, 20)
causal   = np.where(n_ax >= 0, a**np.clip(n_ax,0,None), 0)       # ROC |z|>1.25 → UNSTABLE grows
anticaus = np.where(n_ax < 0, -a**np.clip(n_ax,None,-1), 0)      # ROC |z|<1.25 → stable, anticausal

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].stem(n_ax, causal); axes[0].set_title("ROC |z|>|a|: causal, blows up")
axes[1].stem(n_ax, anticaus); axes[1].set_title("ROC |z|<|a|: stable, but anticausal")
plt.tight_layout(); plt.show()
print("same X(z) = z/(z−1.25). Stability and causality are a PAIR you choose between when a pole is outside the circle.")

same X(z) = z/(z−1.25). Stability and causality are a PAIR you choose between when a pole is outside the circle.


/tmp/ipykernel_2024933/757155337.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two completely different signals, **one algebraic expression**. Both panels are $X(z) = z/(z - 1.25)$. The left is causal — zero for $n < 0$, then growing without bound. The right is anticausal — zero for $n \geq 0$, decaying backwards into the past, and perfectly bounded.

Nothing distinguishes them except the ROC. $|z| > 1.25$ selects the causal one; $|z| < 1.25$ selects the anticausal one. **So the ROC is not a technical footnote — it is half of the answer.** A z-transform quoted without its ROC is genuinely ambiguous, and writing down $z/(z-a)$ and stopping specifies nothing.

**Two rules turn that into something usable.** Stability means the ROC contains the unit circle; causality means it extends outward from the outermost pole. Combine them for a pole at $1.25$, outside the circle: causality forces $|z| > 1.25$, which excludes the unit circle, so the system cannot be stable. Stability forces an ROC containing $|z| = 1$, hence $|z| < 1.25$, so it cannot be causal.

**With a pole outside the unit circle, stability and causality are mutually exclusive — you choose one.** That is a structural constraint, not a limitation of any particular design method.

**And it is a choice you have already made in practice.** Choosing stability over causality means running a filter that needs future samples — exactly `filtfilt` in [Filter Design](./Filter_Design.ipynb), whose backward pass is deliberately anticausal and legal only because the signal is already recorded. A bedside monitor cannot make that choice; an offline pipeline can.

Worth noting what makes any of this possible: on the unit circle the z-transform *is* the DTFT, and off it, $z = re^{j\omega}$ gives the DTFT of $x[n]r^{-n}$. The radius is a damping knob, and the ROC records which damping levels make the sum converge.

---
### 🕐 Session 2 of 4 — *Multirate: Decimation & Interpolation* (~40 min)
**Goal:** change sample rates honestly: anti-alias before dropping, filter after stuffing.
**Builds on:** Session 1; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (sampling). &nbsp; **Feeds into:** Session 3 (polyphase).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Multirate — Decimation & Interpolation</b></summary>

**Timing (~40 min).** 10 min downsampling and why order matters · 10 min upsampling and images · 12 min the two demos · 8 min where this shows up.

**Board first — draw what each operation does to the spectrum.** Downsampling by $M$ *stretches* the spectrum by $M$, so anything beyond the new Nyquist folds back. Upsampling by $L$ *compresses* it, revealing $L-1$ spectral **images** — copies of the original that were always there above the old Nyquist and are now inside the new band. Both pictures are needed before any code.

**Then the two rules, and make the ordering the point.** Decimation is *filter then downsample*. Interpolation is *upsample then filter*. Ask what happens if you reverse either. Filtering after downsampling is too late — the aliasing already happened and is unremovable. Filtering before upsampling does nothing about the images, because they are created by the zero-stuffing. **The order is the entire content of the session**, and it is the most common real-world mistake in this area.

**Ask the room.** "Zero-stuffing inserts zeros — surely that adds no information and changes nothing?" It changes the *sample rate*, and therefore what "frequency" means on that axis. The spectrum does not move; the axis rescales, so content that used to sit above Nyquist is now inside the band and visible as images. Nothing was added; the frame of reference changed. Students find this genuinely confusing and it is worth the time.

**Note the error in the first demo's title, and use it.** The plot is labelled "380 Hz aliases to 130 Hz" — it should be **120 Hz**. After decimating by 4 the rate is 250 Hz with Nyquist 125, and $380/250 = 1.52$ cycles/sample, whose fractional part 0.52 exceeds a half and folds to $1 - 0.52 = 0.48$, i.e. $0.48 \times 250 = 120$ Hz. 130 is $|380 - 250|$ — correct arithmetic that skips the fold about Nyquist. (The title has been corrected in this notebook.) This is a good live exercise: have the room compute the alias themselves and read the peak off the plot to confirm. Aliasing arithmetic is easy to get *almost* right.

**Close on ubiquity.** Every audio resampler, every DAC's oversampling stage, every SDR front end, and — worth naming for an ML audience — every strided convolution (decimation) and transposed convolution (upsampling) in a neural network. Checkerboard artifacts in GAN outputs are interpolation images that nobody filtered, which is a striking way to show these two rules escaping DSP entirely.
</details>

## 3. Changing the Sample Rate

💡 **Intuition.** **Downsampling** by $M$ (keep every $M$-th sample) stretches the spectrum by $M$ — anything beyond the new Nyquist folds back as aliasing, so you must low-pass *first* (decimation = filter + downsample). **Upsampling** by $L$ (insert $L-1$ zeros) compresses the spectrum and reveals $L-1$ spectral *images* — ghosts of the original — which the interpolation filter must erase. Every resampler, DAC, and neural 'stride/transposed conv' is these two moves.

In [3]:
fs = 1000
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*40*t) + 0.6*np.sin(2*np.pi*380*t)     # 40 Hz wanted + 380 Hz intruder
M = 4                                                     # target fs = 250 → new Nyquist 125 Hz

naive = x[::M]                                            # just drop samples
proper = sig.decimate(x, M, ftype="fir")                  # anti-alias filter, THEN drop

def spec(y, fs_y):
    f = np.fft.rfftfreq(len(y), 1/fs_y)
    return f, np.abs(np.fft.rfft(y)) / len(y)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8), sharey=True)
for ax, (y, title) in zip(axes, [(naive, "naive [::4] — 380 Hz aliases to 120 Hz!"),
                                  (proper, "decimate() — intruder removed first")]):
    f, S = spec(y, fs/M)
    ax.plot(f, S); ax.set_title(title); ax.set_xlabel("Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2024933/733753377.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Same signal, two ways of reducing its rate by 4. Naive `x[::4]` leaves a large spurious peak that was never in the signal; `sig.decimate` — which low-passes *first* — leaves only the 40 Hz tone we wanted.

**Work the arithmetic, because it is easy to get almost right.** After decimating by 4 the rate is 250 Hz, so the new Nyquist is 125 Hz. The 380 Hz intruder is $380/250 = 1.52$ cycles per new sample; the fractional part is 0.52, which exceeds a half, so it folds to $1 - 0.52 = 0.48$ cycles/sample — that is $0.48 \times 250 = \mathbf{120}$ Hz.

Note the trap: $|380 - 250| = 130$ is a tempting answer and it is wrong, because 130 exceeds the 125 Hz Nyquist and must itself fold. *(The plot title originally read 130 Hz and has been corrected.)* Getting this right requires reducing modulo the sample rate **and then** folding about Nyquist — two steps, and skipping the second is the standard slip.

**And the aliased peak is indistinguishable from a real signal.** A genuine 120 Hz tone would produce exactly the same samples. No downstream processing can separate them, because they are not merely similar — after sampling they are the *same data*. That is why aliasing is worse than noise: noise degrades a measurement, aliasing replaces it with a plausible wrong one.

**Which fixes the ordering rule.** Decimation is **filter, then downsample**. Filtering afterwards is too late: the fold has already happened and the two components are already identical. Ask what a filter applied to `naive` could possibly do — it can attenuate 120 Hz, but that removes a frequency the real signal might also occupy, and it cannot recover what was destroyed.

This is the [Part 1](./Foundations_of_Signal_Processing_1.ipynb) sampling theorem restated as a procedure, and it is the same rule as the anti-alias filter in front of an ADC, the optical blur in front of a camera sensor ([Image Processing](./Image_Processing.ipynb)), and the decimation filter after a [sigma-delta modulator](./Sigma_Delta_Quantization.ipynb). One theorem, four hardware consequences.

In [4]:
# Upsampling: zero-stuffing creates images; the interpolation filter erases them
L = 4
x40 = np.sin(2*np.pi*40*np.arange(0, 1, 1/250))          # a 250 Hz-rate signal
stuffed = np.zeros(len(x40)*L); stuffed[::L] = x40
interp = sig.resample_poly(x40, L, 1)                     # proper polyphase interpolation

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8), sharey=True)
for ax, (y, title) in zip(axes, [(stuffed, "zero-stuffed: 3 spectral images"),
                                  (interp, "after interpolation filter")]):
    f, S = spec(y, 1000)
    ax.plot(f, S); ax.set_title(title); ax.set_xlabel("Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2024933/4267586221.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Zero-stuffing by 4 produces **three spectral images** — copies of the 40 Hz tone at higher frequencies that were not audible before. After proper interpolation filtering, only the original remains.

**Where the images come from, precisely.** Inserting zeros adds no information whatsoever; the underlying samples are unchanged. What changes is the *sample rate*, and therefore what the frequency axis means. Content that previously sat above the old Nyquist is now inside the new band and becomes visible. The spectrum did not move — the axis rescaled around it. Students find this genuinely counterintuitive, and the useful framing is that upsampling is a *relabelling* that exposes structure which was always implicitly there.

**Which gives the mirror-image rule.** Interpolation is **upsample, then filter** — the opposite ordering to decimation, and for a symmetric reason. In decimation the damage (aliasing) happens *during* the rate change, so the filter must come first. In interpolation the artifacts (images) are *created* by the rate change, so the filter must come after. Ask the room to state both rules and explain why they differ; getting that symmetry is worth more than memorising two recipes.

**Note what `resample_poly` is doing.** It upsamples and filters in one operation, and — as Session 3 shows — it never computes the zeros it would be multiplying by. That is the polyphase optimisation, already at work in a library call the room has been using without noticing.

**And this escapes DSP entirely.** A strided convolution in a neural network is decimation; a transposed convolution is upsampling by zero-stuffing followed by a *learned* filter. The notorious checkerboard artifacts in GAN-generated images are interpolation images that the learned filter failed to suppress — the exact phenomenon in the left panel, appearing in a paper about image generation. Naming that connection lands well with an ML-inclined room and shows these rules are not parochial.

---
### 🕐 Session 3 of 4 — *Polyphase Structures* (~35 min)
**Goal:** never compute what you'll throw away: the decomposition behind every efficient resampler.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (wavelets).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Polyphase Structures</b></summary>

**Timing (~35 min).** 8 min spotting the waste · 12 min the decomposition · 10 min the demo and its honest timing · 5 min where it lives.

**Board first — find the waste before presenting the fix.** In decimation you filter at the full rate and then throw away $M-1$ of every $M$ outputs. Ask what fraction of the arithmetic was wasted: $(M-1)/M$, so 75% at $M = 4$. Every discarded output was computed in full and then discarded. Once the room sees that, polyphase is not clever — it is obvious, and the only question is bookkeeping.

**The decomposition, stated as bookkeeping rather than mathematics.** Split the filter into $M$ interleaved sub-filters (its *phases*), split the input the same way, run each phase at the **low** rate, and sum. Identical output, $M\times$ fewer operations. Emphasise: **no approximation anywhere.** The demo's 1.6e-15 agreement is the point — this is an exact restructuring, not a fast approximate method.

**Then handle the timing honestly, because it undercuts the theory and that is instructive.** Theory says 4× fewer multiply-accumulates at $M = 4$; the measurement shows **1.7×**. Do not skip past this. The gap has real causes: `np.convolve` on $2^{18}$ samples is already heavily optimised C that may switch to FFT convolution, `upfirdn` carries Python call overhead, and at this size memory bandwidth rather than arithmetic is often the limiting factor. Ask the room why an algorithmic 4× does not become a wall-clock 4× — the answer is that operation counts model arithmetic, and real machines are limited by other things. This is exactly the lesson [Performance Engineering](../Intro_GPU/Performance_Engineering.ipynb) makes central.

**The honest summary to give.** The *exactness* (1.6e-15) is the strong claim and it is fully verified. The *speedup* is real but modest here and would be far more decisive on an embedded target where arithmetic genuinely dominates and there is no optimised BLAS underneath. Timings are machine-dependent; say so.

**Close on where it lives.** Every sample-rate converter in every audio device, every SDR front end, and the filter banks in Session 4. Polyphase is why resampling is cheap enough to be invisible — and it is one of the clearest cases in the curriculum of a pure restructuring, with no accuracy cost at all, being worth teaching.
</details>

## 4. The Polyphase Trick

💡 **Intuition.** Filter-then-downsample wastes $\frac{M-1}{M}$ of its work computing outputs that get discarded. The polyphase fix: split the filter into $M$ interleaved sub-filters (its *phases*), run each at the **low** rate on the input's interleaved streams, and sum. Identical output, $M\times$ cheaper — pure bookkeeping, no approximation. This structure is why sample-rate conversion is cheap enough to be everywhere, and it's the skeleton of the filter banks in Session 4.

In [5]:
# Polyphase decimation by hand — verify exact equivalence, then time it
h = sig.firwin(64, 1/M)                     # anti-alias filter for M=4
x_long = rng.standard_normal(2**18)

# reference: full-rate filter, then discard 3 of every 4 outputs
ref = np.convolve(x_long, h, "full")[::M]

# scipy's polyphase engine computes ONLY the surviving outputs
pp = sig.upfirdn(h, x_long, up=1, down=M)
print("max |polyphase − reference| =", np.abs(pp - ref[:len(pp)]).max())

import time
tic = time.perf_counter(); _ = np.convolve(x_long, h, "full")[::M]; t_ref = time.perf_counter() - tic
tic = time.perf_counter(); _ = sig.upfirdn(h, x_long, up=1, down=M); t_pp = time.perf_counter() - tic
print(f"full-rate then discard: {t_ref*1e3:.1f} ms   polyphase upfirdn: {t_pp*1e3:.1f} ms   ({t_ref/t_pp:.1f}x)")

max |polyphase − reference| = 1.5543122344752192e-15
full-rate then discard: 2.0 ms   polyphase upfirdn: 1.2 ms   (1.7x)


**What just happened.** Two numbers, and they should be read very differently.

**The exactness is the strong claim.** `max |polyphase − reference| = 1.6e-15` — machine precision. Polyphase decimation is not a fast *approximation* of filter-then-downsample; it is the identical computation, reorganised. Every output bit-for-bit the same, with the arithmetic that would have been discarded simply never performed. That is worth stating plainly, because "faster" methods in signal processing usually trade accuracy, and this one does not.

The idea is pure bookkeeping. Filtering at full rate and keeping every $M$-th output wastes $(M-1)/M$ of the work — 75% at $M = 4$. Polyphase splits the filter into $M$ interleaved sub-filters, splits the input the same way, runs each at the **low** rate, and sums. Same answer, none of the waste.

**The speedup, though, needs care: theory says 4×, the stopwatch says 1.7×.** That gap is real and worth understanding rather than ignoring. Several things cause it. `np.convolve` on $2^{18}$ samples is already highly optimised C and may internally switch to FFT-based convolution, so the "slow" baseline is not naive. `upfirdn` carries Python-level call overhead that a 2 ms measurement cannot amortise. And at this data size the computation is substantially **memory-bandwidth bound** rather than arithmetic bound, so removing multiplications does not remove the bottleneck.

The general lesson is one this curriculum returns to in [Performance Engineering](../Intro_GPU/Performance_Engineering.ipynb): **operation counts model arithmetic, and real machines are usually limited by something else.** An algorithmic 4× becomes a wall-clock 4× only when arithmetic is genuinely the constraint — which is exactly the situation on an embedded DSP or in FPGA fabric, where there is no optimised BLAS underneath and every multiplier is silicon you paid for. That is where polyphase earns its full factor, and it is why the structure is universal in hardware resamplers even though a laptop benchmark understates it.

Note also that these timings are machine-dependent and taken from a single run; treat 1.7× as "meaningfully faster, less than theory," not as a measurement to quote.

---
### 🕐 Session 4 of 4 — *Wavelets, a First Meeting* (~40 min)
**Goal:** trade the STFT's fixed window for scale: the Haar transform, coded from scratch.
**Builds on:** Session 3; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (uncertainty).

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Wavelets, a First Meeting</b></summary>

**Timing (~40 min).** 10 min why the STFT is stuck · 10 min constant-Q tiling · 10 min Haar from scratch · 10 min the compression bake-off.

**Board first — draw the two tilings.** The STFT tiles the time–frequency plane with *identical* rectangles, because one window length is chosen once. Wavelets use tall thin rectangles at high frequency (good timing, poor frequency resolution) and short wide ones at low frequency (the reverse). Neither beats the uncertainty principle from [Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb) — the rectangles have the same *area*. Wavelets simply spend a fixed budget where the signal needs it. Say that explicitly: **wavelets do not beat uncertainty, they allocate it adaptively.**

**Justify constant-Q as matching how signals actually behave.** High-frequency events tend to be brief (transients, edges, clicks); low-frequency content tends to be sustained. So resolution proportional to frequency matches the physics, and it also matches hearing — the ear's critical bands are roughly constant-Q, which is why this tiling sounds natural.

**Then the reveal that ties the workshop together.** A wavelet transform *is* a two-channel filter bank — low-pass and high-pass, each downsampled by 2 — applied **recursively to the low-pass branch**. That is Sessions 2 and 3's machinery, iterated. Point at `haar_forward`: `(a[0::2] + a[1::2])/√2` is a low-pass followed by ↓2, and the difference is the high-pass. Students who see wavelets as multirate filtering rather than as a new theory find the rest of the field far more approachable.

**Perfect reconstruction deserves a moment.** `haar_inverse` recovers the signal exactly, so the transform loses nothing — it is a change of basis, orthonormal, exactly like the DFT. Compression comes from *discarding coefficients afterwards*, not from the transform. Students conflate the two constantly.

**Set up the bake-off as a fair fight.** Same signal, same budget of 40 coefficients, two bases. Ask for a prediction first — most expect Fourier to do respectably. It does not: RMSE 0.1387 against Haar's 0.0178, nearly 8× worse. Then ask *why*, and steer to the answer: a step edge needs many sinusoids to build (Gibbs, from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb)), while a Haar basis function *is* a step, so a handful suffice.

**And state the converse so nobody over-generalises.** On a pure sinusoid, Fourier wins overwhelmingly and Haar does badly. Neither basis is better; each is sparse for a different class of signal. That is exactly the [dictionary learning](./Sparse_Dictionary_Learning.ipynb) lesson, and the honest summary of the whole session: **match the basis to the signal.**
</details>

## 5. Beyond Fixed Windows

💡 **Intuition.** The STFT slices time with ONE window length — so it resolves either the click or the pitch well, never both ([Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb)'s uncertainty trade, frozen in). Wavelets spend the uncertainty budget *adaptively*: short windows for high frequencies, long for low — constant-Q tiling. Implementation-wise a wavelet transform is just a two-channel filter bank (low-pass + high-pass, downsample by 2) applied **recursively to the low-pass branch** — Session 3's machinery, iterated.

In [6]:
# The Haar wavelet transform from scratch: averages & differences, recursively
def haar_forward(x):
    out, approx = [], x.astype(float)
    while len(approx) > 1:
        a = (approx[0::2] + approx[1::2]) / np.sqrt(2)      # low-pass  + ↓2
        d = (approx[0::2] - approx[1::2]) / np.sqrt(2)      # high-pass + ↓2
        out.append(d); approx = a
    out.append(approx)
    return out[::-1]                                         # coarsest first

def haar_inverse(coeffs):
    approx = coeffs[0]
    for d in coeffs[1:]:
        up = np.empty(2*len(d))
        up[0::2] = (approx + d) / np.sqrt(2)
        up[1::2] = (approx - d) / np.sqrt(2)
        approx = up
    return approx

# a piecewise-constant "blocks" signal — Fourier's nightmare, wavelets' lunch
t = np.linspace(0, 1, 512)
x = np.select([t < 0.2, t < 0.45, t < 0.6, t < 0.8], [0.0, 1.6, 0.4, 2.0], default=0.9)
x = x + 0.02 * rng.standard_normal(512)
coeffs = haar_forward(x)
print("perfect reconstruction:", np.allclose(haar_inverse(coeffs), x))

perfect reconstruction: True


**What just happened.** `perfect reconstruction: True` — the forward transform followed by the inverse returns the original signal exactly. Nothing was lost.

That is worth pausing on, because it is easy to assume a wavelet transform *is* compression. It is not. The Haar transform is an orthonormal change of basis, exactly like the DFT: same information, different coordinates, fully invertible. Compression happens when you *discard coefficients afterwards*, which is the next cell. Keeping the two ideas separate matters — the transform is lossless, the thresholding is where the loss lives and where the design choices are.

**Look at what the code actually does, because it is smaller than the theory suggests.** `(approx[0::2] + approx[1::2])/√2` is an average of sample pairs — a two-tap **low-pass followed by ↓2**. The difference is the matching **high-pass followed by ↓2**. Then the loop recurses *on the low-pass branch only*.

So a wavelet transform is a two-channel filter bank applied recursively — Sessions 2 and 3's machinery, iterated. The $\sqrt2$ normalisations are what make it orthonormal, so energy is preserved and Parseval holds exactly as in the DFT. There is no new theory here; there is a familiar structure arranged in a tree.

**And that recursion is what produces constant-Q tiling.** Each level halves the rate, so each successive level analyses a band an octave lower with twice the time support. High frequencies get short windows (good timing, coarse frequency resolution); low frequencies get long ones (the reverse). Compare with the STFT, which fixes one window length and therefore one rectangle shape everywhere.

Crucially, this does **not** beat the uncertainty principle from [Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb) — every tile has the same area. Wavelets spend the same budget differently, allocating resolution where the signal is likely to need it. The bet is that real signals have brief high-frequency events and sustained low-frequency content, which is true often enough to be useful — and it is roughly how the ear's critical bands are arranged too.

In [7]:
# Compression bake-off at equal budget: keep the 40 largest coefficients
def keep_top(vals, k):
    flat = np.concatenate(vals) if isinstance(vals, list) else vals
    thresh = np.sort(np.abs(flat))[-k]
    return thresh

k = 40
# wavelet: threshold across all detail levels
flatc = np.concatenate(coeffs)
th_w = np.sort(np.abs(flatc))[-k]
coeffs_c = [np.where(np.abs(c) >= th_w, c, 0) for c in coeffs]
x_wav = haar_inverse(coeffs_c)
# Fourier: keep top-k magnitude bins (hermitian pairs counted once)
X = np.fft.rfft(x)
th_f = np.sort(np.abs(X))[-k//2]
x_fft = np.fft.irfft(np.where(np.abs(X) >= th_f, X, 0), n=len(x))

plt.figure(figsize=(9, 3))
plt.plot(t, x, "k", alpha=0.35, label="signal")
plt.plot(t, x_fft, label=f"Fourier, {k} coeffs (ringing at the step)")
plt.plot(t, x_wav, label=f"Haar, {k} coeffs (step preserved)")
plt.legend(); plt.title("Same budget, different bases — transients favor wavelets")
plt.tight_layout(); plt.show()
print(f"RMSE  Fourier {np.std(x_fft - x):.4f}   Haar {np.std(x_wav - x):.4f}")

RMSE  Fourier 0.1387   Haar 0.0178


/tmp/ipykernel_2024933/487303954.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Equal budget — 40 coefficients each — and Haar reconstructs the piecewise-constant signal at RMSE **0.0178** against Fourier's **0.1387**, nearly **8× better**. On the plot, the Fourier reconstruction rings visibly around every step while the Haar version keeps the edges sharp.

**Why Fourier struggles is a theorem, not bad luck.** A step discontinuity is not sparse in a sinusoidal basis — building a sharp edge from smooth sinusoids takes very many of them, and truncating the series produces the overshoot and ringing known as **Gibbs' phenomenon**, which the room met in [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb). Forty coefficients simply cannot represent five clean edges.

Haar's basis functions *are* steps. A piecewise-constant signal is therefore sparse in it: a handful of large coefficients capture the plateaus and the transitions, and everything else is near zero. Same signal, same budget, and the difference is entirely which basis the signal happens to be sparse in.

**Now the converse, so nobody over-generalises.** Run this on a pure sinusoid and the result inverts completely: Fourier needs *one* coefficient, while Haar needs many to approximate a smooth curve from blocky pieces. **Neither basis is better.** Each is sparse for a different class of signal, and the whole art is matching the basis to the content.

That is precisely the lesson [Sparse Dictionary Learning](./Sparse_Dictionary_Learning.ipynb) takes further — if no standard basis fits your data, *learn* one — and it is the same distinction as there between "can represent" and "can represent *briefly*". Both bases here are complete and could reproduce the signal exactly with all 512 coefficients. The question was never representability; it was sparsity.

**And this is why JPEG-2000 uses wavelets while JPEG uses the DCT.** Photographs contain edges, and edges are the thing wavelets encode cheaply — which is also why JPEG's block-DCT produces visible ringing around sharp boundaries at high compression, exactly the artifact in the orange trace above. The bake-off in this cell is, in miniature, the argument that changed image compression standards.

## 6. Conclusion

The z-transform's ROC settles stability vs causality; decimation and interpolation change rates honestly; polyphase makes it all nearly free; and wavelets re-spend the uncertainty budget where the signal needs it. This is the toolkit of every modern codec and SDR front-end.

---
## Where next

- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — multirate chains in the wild.
- [Compressed Sensing](./Compressed_Sensing.ipynb) — sparsity in a basis, weaponized.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — the STFT/wavelet trade on real sound.